<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the libraries we can get from pip in one line, which covers data handling and the performance report.

In [ ]:
!pip install pandas pyfolio-reloaded

Zipline is left out on purpose because it needs more than a pip command. Install it with conda (conda install -c conda-forge zipline-reloaded) so the compiled dependencies resolve, then ingest the "quandl-eod" price bundle before you run anything, because the test reads prices from that bundle and not from the internet.

## Imports and setup

We use pandas for the start and end timestamps, zipline to run the strategy against historical prices, and pyfolio to turn the result into a performance and risk report.

In [ ]:
import warnings

In [ ]:
import pandas as pd
import pyfolio as pf
from zipline import run_algorithm
from zipline.api import (
    attach_pipeline,
    date_rules,
    order_target_percent,
    pipeline_output,
    record,
    schedule_function,
    symbol,
    time_rules,
    get_open_orders,
)
from zipline.finance import commission, slippage
from zipline.pipeline import Pipeline
from zipline.pipeline.factors import SimpleMovingAverage
from zipline.pipeline.data import USEquityPricing

Zipline and pyfolio both print deprecation notices from older pandas calls, and we silence them here.

In [ ]:
warnings.filterwarnings("ignore")

Those warnings have nothing to do with our results, and a wall of red text on the first run makes people think the strategy failed. Turn this off again if you start editing the strategy logic, because real errors are easier to spot without a filter in the way.

## Set up the strategy rules

This function runs once before the test starts. It names the five funds we can hold, sets the length of the long-term average, schedules the monthly decision, and sets our trading costs.

In [ ]:
def initialize(context):
    context.symbols = [
        symbol("SPY"),
        symbol("EFA"),
        symbol("IEF"),
        symbol("VNQ"),
        symbol("GSG"),
    ]
    context.sma = {}
    context.period = 10 * 21

    for asset in context.symbols:
        context.sma[asset] = SimpleMovingAverage(
            inputs=[USEquityPricing.close],
            window_length=context.period,
        )

    schedule_function(
        func=rebalance,
        date_rule=date_rules.month_start(),
        time_rule=time_rules.market_open(minutes=1),
    )

    context.set_commission(
        commission.PerShare(cost=0.01, min_trade_cost=1.00)
    )
    context.set_slippage(slippage.VolumeShareSlippage())

The 10 * 21 is 210 trading days, which is roughly the 10-month average Faber used, since a month holds about 21 trading days. The commission and slippage settings matter more than beginners expect, because commission is the fee we pay per share and slippage is the gap between the price we saw and the price we got. Leave both out and a strategy that trades often can look profitable when it isn't.

## Rebalance the funds each month

This runs on the first market day of each month. It builds the list of funds trading above their own long-term average, sells anything that isn't on the list, and splits our money evenly across the ones that are.

In [ ]:
def rebalance(context, data):
    longs = [
        asset
        for asset in context.symbols
        if data.current(asset, "price") > context.sma[asset].mean()
    ]

    for asset in context.portfolio.positions:
        if asset not in longs and data.can_trade(asset):
            order_target_percent(asset, 0)

    for asset in longs:
        if data.can_trade(asset):
            order_target_percent(asset, 1.0 / len(longs))

Notice we sell first and buy second, so the cash from the exits is available for the new positions. If only two funds sit above their average, each gets 50% and the other 60% of our money stays in cash, which is where the smaller losses come from. The can_trade check keeps us from sending orders on days a fund has no price, which is the kind of small guard that saves you an afternoon of debugging.

## Run the test and read results

Here we run the strategy over historical prices from 2010 through June 2023 with $100,000 of starting capital, reading daily prices from the "quandl-eod" bundle.

In [ ]:
start = pd.Timestamp("2010")
end = pd.Timestamp("2023-06-30")

In [ ]:
perf = run_algorithm(
    start=start,
    end=end,
    initialize=initialize,
    capital_base=100000,
    bundle="quandl-eod",
)

The perf object holds one row per trading day with our portfolio value, positions, and every order filled. Zipline only ever hands the strategy prices that existed on the day it makes a decision, so we can't accidentally use tomorrow's price to make today's trade. That single property is why we bother with a library instead of comparing columns in a spreadsheet.

This pulls the three pieces pyfolio needs out of the daily results, which are our returns, what we held, and what we traded.

In [ ]:
returns, positions, transactions = (
    pf.utils.extract_rets_pos_txn_from_zipline(perf)
)

Returns alone tell us how the money grew, but the positions and transactions let pyfolio show where the money sat and what the trading cost us. Keeping the three separate is also useful when you want to check one fund's contribution by hand.

This prints the full performance and risk report, including the equity curve, the worst peak-to-trough losses, and a breakdown of each completed round trip.

In [ ]:
pf.create_full_tear_sheet(
    returns,
    positions=positions,
    transactions=transactions,
    round_trips=True,
)

Start with the worst peak-to-trough loss table, which shows how far our account fell from a previous high and how long it took to recover, because that 28% figure is the part you'd actually have to live through. The round_trips=True flag adds statistics on each completed buy-and-sell pair, so we can see how many of the monthly holds made money and how long the average one lasted. Read the report before you touch the rules, since most first attempts fail on the size of the losses rather than the total return.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.